In [ ]:
import os
import numpy as np 
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler

In [ ]:
data_dir = '/kaggle/input/chest_xray' 

train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')
val_dir = os.path.join(data_dir, "val")



In [ ]:
from torchvision import transforms

data_transform = {

    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}


In [ ]:
from torch.utils.data import random_split

# Let's have train + test folders
full_train_dataset = datasets.ImageFolder(train_dir, transform=data_transform['train'])


train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size


train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])  # random_split is a function from torch.utils.data that takes: 1. A dataset (like your custom dataset, ImageFolder, etc.) 2. Lengths of the splits you want (e.g., 80/20 or 70/20/10)


image_datasets = {
    'train': train_dataset,
    'val': val_dataset,
    'test': datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=data_transform['test'])
}


dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}

for x in ['train', 'val', 'test']:
    print("Loaded {} images under {}".format(dataset_sizes[x], x))


In [ ]:
# Get all class indices (e.g., 0 for NORMAL, 1 for PNEUMONIA) . we are performing this operation to find count how many samples each class has, so that weights compute class weights
# note _ means “ignore this variable” (the file path). 

# Access underlying dataset from the Subset
train_dataset = image_datasets['train']

# The Subset stores the indices of samples from the original ImageFolder dataset
train_targets = [train_dataset.dataset.imgs[i][1] for i in train_dataset.indices]

# Convert to tensor 
train_targets = torch.tensor(train_targets)

# Count number of samples per class (e.g., 0 = NORMAL, 1 = PNEUMONIA)
class_counts = torch.bincount(train_targets)

# Compute class weights (inverse of counts)
class_weights = 1. / class_counts.float()

# Assign weight to each sample according to its class
sample_weights = [class_weights[label] for label in train_targets]

# Create sampler
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# Training dataloader with weighted sampler for class imbalance
train_dataloader = torch.utils.data.DataLoader(
    image_datasets['train'],
    batch_size=32,
    sampler=sampler,                                        # Only use sampler for training
    num_workers=4
)

# Validation dataloader (no sampler, no shuffle needed)
val_dataloader = torch.utils.data.DataLoader(
    image_datasets['val'],
    batch_size=32,
    shuffle=False,                                          # Don't shuffle validation data
    num_workers=4
)

# Test dataloader (no sampler, no shuffle)
test_dataloader = torch.utils.data.DataLoader(
    image_datasets['test'],
    batch_size=32,
    shuffle=False,                                          # Don't shuffle test data
    num_workers=4
)

# Combine into dictionary
dataloader = {
    'train': train_dataloader,
    'val': val_dataloader,
    'test': test_dataloader
}


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torchvision import models
import torch.nn as nn

# Load pretrained ResNet50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Change last FC layer (for 2 classes)
num_classes = 2
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

# Move model to GPU/CPU
model = model.to(device)

print("Model loaded!")


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Label smoothing helps generalization
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW is usually better than Adam for vision models
learning_rate = 1e-4   # slightly higher than before since we use scheduler
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

num_epochs = 20  # You wanted 20 epochs

# Cosine annealing LR schedule over full training
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


In [ ]:

from tqdm import tqdm

def train_model(model, dataloader, criterion, optimizer, scheduler,
                num_epochs=20, best_model_path="resnet50_xray_best.pth"):
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 20)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            total = 0

            for inputs, labels in tqdm(dataloader[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    with torch.cuda.amp.autocast():   # AMP autocast
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)

                    _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)
                total += labels.size(0)

            epoch_loss = running_loss / total
            epoch_acc = running_corrects.float() / total

            print(f"{phase} Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

            # ✅ Save best model whenever validation accuracy improves
            if phase == "val":
                if epoch_acc > best_val_acc:
                    best_val_acc = epoch_acc
                    torch.save(model.state_dict(), best_model_path)
                    print(f"✅ New best model saved (val acc: {best_val_acc:.4f})")

        # step LR after each epoch
        scheduler.step()

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    # Load the best model weights before returning
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    return model
model = train_model(model, dataloader, criterion, optimizer, scheduler,
                    num_epochs=num_epochs, best_model_path="resnet50_xray_best.pth")
